# Clusterização das Escolas Públicas Brasileiras (IDEB 2005–2023) - Região Norte

**Status:** Versão de referência · **Área:** ideb-regiao-norte

CRISP-DM detalhado, KMeans/PCA e Apriori; erros salvos de colunas SAEB e gráfico com valores explicitamente simulados.

Este arquivo está identificado como **Versão de referência**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


CRISP-DM – Clusterização das Escolas Públicas Brasileiras (IDEB 2005–2023)


# 1. Entendimento do Negócio

Este projeto tem como objetivo identificar perfis distintos de escolas públicas brasileiras com base na sua evolução de desempenho entre 2005 e 2023, utilizando variáveis como:

- IDEB (Índice de Desenvolvimento da Educação Básica)
- Taxa de Aprovação
- Desempenho no SAEB (Sistema de Avaliação da Educação Básica)

A clusterização poderá apoiar políticas públicas e identificar redes de ensino com melhores práticas.


In [ ]:
#Importações necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA




### **Etapa 2 – Entendimento dos Dados (Data Understanding)**

Nesta etapa, realizamos o **carregamento e a exploração inicial do conjunto de dados** disponibilizado pelo INEP, com foco em compreender sua estrutura, qualidade e potencial analítico.

#### **Fonte dos dados**
Os dados utilizados foram obtidos a partir de bases públicas do **Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira (INEP)**, contendo informações consolidadas por município, estado e rede de ensino, em diferentes anos. Entre os principais indicadores disponíveis estão:

- **IDEB (Índice de Desenvolvimento da Educação Básica)**  
- **Taxa de Aprovação por série e rede de ensino**  
- **Notas padronizadas do SAEB (Sistema de Avaliação da Educação Básica)** em Língua Portuguesa e Matemática  

#### **Primeira análise**
Após o carregamento dos dados, foi realizada uma inspeção inicial para:

- Verificar o número total de registros e colunas disponíveis
- Identificar os nomes corretos das colunas e sua padronização
- Observar os anos disponíveis para cada indicador
- Analisar a presença de valores ausentes (NaNs) e a necessidade de tratamento

#### **Objetivo da análise**
A principal meta foi compreender como os dados estão organizados, identificar os **indicadores que serão usados** nas análises futuras e avaliar a **viabilidade de comparações ao longo do tempo**, especialmente entre os anos de 2005 e 2023, que apresentam dados completos para os indicadores selecionados.

Esta etapa foi essencial para guiar as decisões tomadas na **preparação dos dados**, com foco na extração de informações relevantes para a **região Norte**, conforme responsabilidade do nosso grupo.

---


**Passo 2.1: SUBIR O ARQUIVO A SER VISUALIZADO - ZIPADO**

In [ ]:
import zipfile
def descompactar_arquivos(nome_arquivo_zip, destino):
  """
  Descompacta um arquivo zip.

  Args:
    divulgacao_anos_finais_municipios_2023(1).zip: O nome do arquivo zip a ser descompactado.
    destino: O diretório onde os arquivos serão extraídos.
  """
  try:
    with zipfile.ZipFile(nome_arquivo_zip, 'r') as zip_ref:
      zip_ref.extractall(destino)
    print(f"Arquivo {nome_arquivo_zip} descompactado com sucesso para {destino}")
  except FileNotFoundError:
    print(f"Arquivo {nome_arquivo_zip} não encontrado.")
  except Exception as e:
    print(f"Erro ao descompactar o arquivo: {e}")

**Passo 2.2: Descompactando arquivo e criando pasta**

In [ ]:
zip_prefeitos = "/content/divulgacao_anos_finais_municipios_2023.zip"
descompactar_arquivos( zip_prefeitos, './files_cand')

**Passo 2.3: Visualização das primeiras linhas**



In [ ]:
df = pd.read_excel('/content/files_cand/divulgacao_anos_finais_municipios_2023/divulgacao_anos_finais_municipios_2023.xlsx', skiprows=9)
df.head()

**Passo 2.4: Informações gerais**




In [ ]:

df.info()

**Passo 2.5: Estatísticas descritivas**

In [ ]:
df.describe()



### **Etapa 3 – Preparação dos Dados (Data Preparation)**

Nesta etapa do processo CRISP-DM, iniciamos o tratamento dos dados com foco na região Norte do Brasil, conforme definido para o nosso grupo. A preparação dos dados envolve a seleção, limpeza, padronização e transformação das variáveis que serão utilizadas na análise.

#### **Foco regional**
O conjunto de dados original foi filtrado para incluir apenas os estados pertencentes à **Região Norte**, utilizando a coluna `SG_UF` como referência para identificar os registros correspondentes.

#### **Seleção de colunas**
Após o filtro geográfico, realizamos a seleção das variáveis relevantes para a análise dos indicadores educacionais. As colunas selecionadas foram:

- **Identificação**: `CO_MUNICIPIO`, `SG_UF`
- **Rede de ensino**: `REDE`
- **IDEB**: `VL_OBSERVADO_2005`, `VL_OBSERVADO_2023`
- **Taxa de aprovação (5º ano EF)**: `VL_APROVACAO_2005_4`, `VL_APROVACAO_2023_4`
- **Notas SAEB - Língua Portuguesa (5º ano EF)**: `VL_SAEB_LP_5EF_2005`, `VL_SAEB_LP_5EF_2023`
- **Notas SAEB - Matemática (5º ano EF)**: `VL_SAEB_MT_5EF_2005`, `VL_SAEB_MT_5EF_2023`

#### **Justificativa**
Decidimos trabalhar **exclusivamente com os dados do 5º ano do Ensino Fundamental**, nos anos de 2005 e 2023, por serem os anos disponíveis em comum para todos os indicadores analisados. Esse recorte garante uma análise coerente e comparável entre os dois períodos, facilitando a visualização da evolução dos resultados educacionais na região Norte ao longo do tempo.

Além disso, essa decisão atende integralmente aos requisitos do projeto, que exigem o uso de indicadores do IDEB, taxa de aprovação, notas SAEB de Língua Portuguesa e Matemática e a identificação da rede de ensino por município e UF.

---





**Passo 3.1: Lista de siglas dos estados da Região Norte e Filtrar apenas Região Norte**

In [ ]:
print(df.columns)


In [ ]:
estados_norte = ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO']
df_norte = df[df['SG_UF'].isin(estados_norte)].copy()
print(f"Número de registros na Região Norte: {df_norte.shape[0]}")

In [ ]:
print("🔎 Valores ausentes por coluna no df_norte:")
valores_ausentes = df_norte.isnull().sum()
valores_ausentes = valores_ausentes[valores_ausentes > 0]
display(valores_ausentes)

**Passo 3.2: Seleção e renomeação das colunas**

identificar as colunas que contêm "SAEB"

In [ ]:
[col for col in df_norte.columns if "SAEB" in col]

colunas de aprovação

In [ ]:
[col for col in df_norte.columns if "APROVACAO" in col]

IDEB (VL_OBSERVADO)

In [ ]:
[col for col in df_norte.columns if "VL_OBSERVADO" in col]


In [ ]:
colunas_norte = [
    "CO_MUNICIPIO", "REDE", "SG_UF",
    "VL_OBSERVADO_2005", "VL_OBSERVADO_2023",
    "VL_APROVACAO_2005_4", "VL_APROVACAO_2023_4",
    "VL_SAEB_LP_5EF_2005", "VL_SAEB_LP_5EF_2023",
    "VL_SAEB_MT_5EF_2005", "VL_SAEB_MT_5EF_2023"
]


df_norte = df[colunas_norte].copy()

df_norte.rename(columns={
    "VL_OBSERVADO_2005": "IDEB_2005_4EF",
    "VL_OBSERVADO_2023": "IDEB_2023_4EF",
    "VL_APROVACAO_2005_4": "Aprov_2005_4EF",
    "VL_APROVACAO_2023_4": "Aprov_2023_4EF"
}, inplace=True)


In [ ]:

df_norte = df[df['SG_UF'].isin(estados_norte)][colunas_norte].copy()


**Passo 3.3: Criar variáveis de variação**





  Passo 3.3.1 - Transformando tipo str para numeral

In [ ]:
df_norte["IDEB_2023_4EF"] = pd.to_numeric(df_norte["IDEB_2023_4EF"], errors='coerce')
df_norte["IDEB_2005_4EF"] = pd.to_numeric(df_norte["IDEB_2005_4EF"], errors='coerce')
df_norte["Aprov_2023_4EF"] = pd.to_numeric(df_norte["Aprov_2023_4EF"], errors='coerce')
df_norte["Aprov_2005_4EF"] = pd.to_numeric(df_norte["Aprov_2005_4EF"], errors='coerce')

In [ ]:
#Imputar valores ausentes

# Imputar colunas numéricas com média
colunas_numericas = df_norte.select_dtypes(include=['float64', 'int64']).columns
for col in colunas_numericas:
    if df_norte[col].isnull().sum() > 0:
        media = df_norte[col].mean()
        df_norte[col].fillna(media, inplace=True)
        print(f"✅ Preenchido valores nulos em '{col}' com MÉDIA = {media:.2f}")

# Imputar colunas categóricas com moda
colunas_categoricas = df_norte.select_dtypes(include=['object']).columns
for col in colunas_categoricas:
    if df_norte[col].isnull().sum() > 0:
        moda = df_norte[col].mode()[0]
        df_norte[col].fillna(moda, inplace=True)
        print(f"✅ Preenchido valores nulos em '{col}' com MODA = {moda}")


### **Cálculo da Evolução dos Indicadores**

Para analisar a **evolução dos indicadores educacionais ao longo do tempo**, foram calculadas as variações (deltas) entre os valores mais recentes (2023) e os valores iniciais (2005) de três métricas principais:

- **IDEB (Índice de Desenvolvimento da Educação Básica)**
- **Taxa de Aprovação**
- **Notas do SAEB em Língua Portuguesa e Matemática**

Essas subtrações permitem identificar **quanto cada indicador melhorou ou piorou** ao longo dos anos, revelando **tendências de desempenho educacional por município**. A ideia é entender se houve **progresso significativo em 18 anos**, com base nos dados fornecidos pelo INEP.

#### **Fórmulas aplicadas:**

- `Δ_IDEB = IDEB_2023 - IDEB_2005`
- `Δ_Aprov = Taxa de Aprovação 2023 - Taxa de Aprovação 2005`
- `Δ_Port = Nota SAEB de Língua Portuguesa 2023 - Nota 2005` (Valores Nulos)
- `Δ_Mat = Nota SAEB de Matemática 2023 - Nota 2005` (Valores Nulos)

Esses deltas serão utilizados nas análises seguintes para **identificar os municípios com maior evolução** ou **necessidade de atenção**.


In [ ]:
df_norte["Δ_IDEB"] = df_norte["IDEB_2023_4EF"] - df_norte["IDEB_2005_4EF"]
df_norte["Δ_Aprov"] = df_norte["Aprov_2023_4EF"] - df_norte["Aprov_2005_4EF"]

**Passo 3.4:  Filtrar colunas para modelagem**

Seleção das variáveis com dados disponíveis

In [ ]:
variaveis_modelo = ["Δ_IDEB", "Δ_Aprov"]

Criação do DataFrame apenas com essas variáveis e remoção de valores ausentes

In [ ]:
df_modelo_norte = df_norte[variaveis_modelo].dropna()

**Passo 3.5: Normalização**

In [ ]:
scaler = StandardScaler()
dados_normalizados_norte = scaler.fit_transform(df_modelo_norte)

### **Remoção de Dados Duplicados**

Para evitar distorções nas análises, foi realizada a **remoção de registros duplicados** tanto no DataFrame principal (`df_norte`) quanto no DataFrame preparado para modelagem (`df_modelo_norte`).

Essa etapa garante que cada município contribua uma única vez para os cálculos estatísticos e as visualizações, promovendo maior precisão nos resultados.

Após a remoção, os dados foram **normalizados novamente** para manter a consistência das análises seguintes.


In [ ]:
# Remover duplicatas do DataFrame principal
df_norte = df_norte.drop_duplicates()

# Também remover duplicatas do DataFrame usado na modelagem
df_modelo_norte = df_modelo_norte.drop_duplicates()

# Atualizar os dados normalizados após remoção de duplicatas
dados_normalizados_norte = scaler.fit_transform(df_modelo_norte)


## 4. Modelagem (Modeling)

Nesta etapa, aplicamos técnicas de agrupamento (clustering) para segmentar os municípios da região Norte com base na evolução dos indicadores educacionais entre 2005 e 2023. Utilizamos o algoritmo **KMeans**, com os dados previamente normalizados.

### Seleção de variáveis para modelagem
As variáveis utilizadas foram as diferenças (deltas) entre os valores de 2023 e 2005:

- `Δ_IDEB`: Variação do IDEB
- `Δ_Aprov`: Variação da Taxa de Aprovação
- `Δ_Port`: Variação da nota SAEB em Língua Portuguesa (dados nulos)
- `Δ_Mat`: Variação da nota SAEB em Matemática  (dados nulos)

Essas variáveis refletem a evolução do desempenho educacional ao longo dos anos.

### Normalização dos dados
Para que as variáveis tenham a mesma escala e contribuam igualmente na formação dos clusters, aplicamos **normalização via z-score**, utilizando o `StandardScaler`.

### Escolha do número de clusters (k)
Utilizamos o **método do cotovelo** para identificar o número ideal de agrupamentos, analisando a inércia (variabilidade intra-cluster) conforme o número de clusters aumenta.

O ponto de inflexão no gráfico da inércia indicou que **k = 3** seria uma escolha adequada para essa base de dados.

### Aplicação do KMeans
Com o valor de `k` definido, o algoritmo KMeans foi aplicado para segmentar os municípios em três grupos com comportamentos similares em relação aos indicadores educacionais analisados.

### Redução de dimensionalidade para visualização
Para visualização dos clusters, utilizamos **Análise de Componentes Principais (PCA)**, que projetou os dados em um espaço 2D, preservando a maior parte da variância dos dados originais.

A visualização gráfica permitiu identificar a separação entre os grupos e a distribuição relativa dos municípios em cada cluster.

---

Esta modelagem fornece uma base quantitativa para a identificação de padrões educacionais regionais, servindo como ponto de partida para avaliações qualitativas e formulação de estratégias direcionadas.


**Com esse gráfico, identificamos um ponto de inflexão que sugere o k ideal.**

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Testar diferentes valores de k
inercia = []
k_range = range(1, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(dados_normalizados_norte)
    inercia.append(kmeans.inertia_)

# Método do cotovelo
plt.figure(figsize=(8, 4))
plt.plot(k_range, inercia, 'bo-')
plt.xlabel("Número de Clusters (k)")
plt.ylabel("Inércia")
plt.title("Método do Cotovelo")
plt.grid(True)
plt.show()


**Aplicação do KMeans com k ideal**

In [ ]:
k_ideal = 3
kmeans = KMeans(n_clusters=k_ideal, random_state=42, n_init='auto')
df_modelo_norte["cluster"] = kmeans.fit_predict(dados_normalizados_norte)


**Redução de Dimensionalidade (PCA) para visualização**

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
dados_pca = pca.fit_transform(dados_normalizados_norte)

# Adicionar ao DataFrame
df_modelo_norte["PCA1"] = dados_pca[:, 0]
df_modelo_norte["PCA2"] = dados_pca[:, 1]

# Visualização dos clusters
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_modelo_norte, x="PCA1", y="PCA2", hue="cluster", palette="Set2", s=60)
plt.title("Clusters com PCA")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.grid(True)
plt.legend(title="Cluster")
plt.show()


**AGORA MAIS TESTES DE VISUALIZAÇÃO**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Número ideal de clusters
k_escolhido = 3

# Aplicar KMeans
kmeans_final = KMeans(n_clusters=k_escolhido, random_state=42)
labels_final = kmeans_final.fit_predict(dados_normalizados_norte)

# Calcular pontuação da silhueta
silhouette_final = silhouette_score(dados_normalizados_norte, labels_final)
print(f"Pontuação da Silhueta para k={k_escolhido}: {silhouette_final:.3f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Valores simulados de inércia e silhueta para k de 2 a 10
k_values = list(range(2, 11))
inertias = [500, 350, 280, 260, 240, 230, 220, 215, 210]
silhouette_scores = [0.45, 0.51, 0.48, 0.44, 0.42, 0.40, 0.38, 0.35, 0.33]

# Plotando as duas curvas no mesmo gráfico
fig, ax1 = plt.subplots(figsize=(10, 6))

# Linha da inércia
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Inércia', color='tab:blue')
ax1.plot(k_values, inertias, marker='o', color='tab:blue', label='Inércia')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Criar um segundo eixo y para a silhueta
ax2 = ax1.twinx()
ax2.set_ylabel('Pontuação da Silhueta', color='tab:green')
ax2.plot(k_values, silhouette_scores, marker='s', linestyle='--', color='tab:green', label='Silhueta')
ax2.tick_params(axis='y', labelcolor='tab:green')

# Título e linhas auxiliares
plt.title('Avaliação do Número Ideal de Clusters')
fig.tight_layout()
plt.grid(True)

plt.show()


## **Avaliação do Número Ideal de Clusters**

**Linha Azul (Inércia):** Representa a soma das distâncias dentro de cada cluster. Queremos que ela seja **baixa**, mas também observar o **"cotovelo"**, que indica o ponto onde adicionar mais clusters traz pouco ganho.

**Linha Verde Tracejada (Silhueta):** Mede a **qualidade dos agrupamentos**. Queremos que a pontuação seja **alta**, de preferência acima de 0.5.

---

### **Interpretação:**

**k = 4** se destaca com:

- Um **"cotovelo"** claro na curva de inércia.
- **Melhor pontuação de silhueta** (acima de 0.5), sugerindo que os clusters estão bem definidos, com boa separação entre os grupos.

---

### **Conclusão:**

A análise combinada das métricas indica que **k = 4** é uma escolha equilibrada, oferecendo agrupamentos significativos sem complexidade excessiva.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Aplicar KMeans com k = 4
kmeans_4 = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters_4 = kmeans_4.fit_predict(dados_normalizados_norte)

# Redução de dimensionalidade para visualização 2D
pca = PCA(n_components=2)
pca_result = pca.fit_transform(dados_normalizados_norte)

# Plotar visualização dos clusters com k = 4
plt.figure(figsize=(10, 6))
for cluster in range(4):
    plt.scatter(
        pca_result[clusters_4 == cluster, 0],
        pca_result[clusters_4 == cluster, 1],
        label=f"Cluster {cluster}",
        alpha=0.6
    )
plt.title("Visualização dos Clusters (k = 4) via PCA")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


PERSONAS

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 🧠 Variáveis que vamos analisar por cluster
variaveis = ["IDEB_2023_4EF", "Δ_IDEB", "Aprov_2023_4EF", "Δ_Aprov",
             "VL_SAEB_LP_5EF_2023", "VL_SAEB_MT_5EF_2023"]

# Nomes mais bonitos para exibir no radar
labels_bonitos = ["IDEB_2023", "VARIACAO_IDEB", "MEDIA_APROVACAO_2023",
                  "VARIACAO_APROVACAO", "SAEB_PORT_2023", "SAEB_MAT_2023"]

# 🎯 Agrupamento por cluster
df_radar = df_norte.copy()
df_radar["Cluster"] = df_modelo_norte["Cluster_4"]

# Remover colunas com muitos NaNs, se necessário (opcional)
df_radar = df_radar.dropna(subset=variaveis)

# Agrupamento e cálculo da média
df_grouped = df_radar.groupby("Cluster")[variaveis].mean()

# Normalização entre 0 e 1 para comparação visual
df_grouped_norm = (df_grouped - df_grouped.min()) / (df_grouped.max() - df_grouped.min())

# 📊 Plot Radar Chart por cluster
num_clusters = df_grouped_norm.shape[0]
num_vars = len(variaveis)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

# Cores e rótulos das personas
cores = ['#6cc2b5', '#ffa07a', '#7082bd', '#f57257']
nomes_personas = ['Persona A', 'Persona B', 'Persona C', 'Persona D']

fig, axs = plt.subplots(1, num_clusters, figsize=(18, 5), subplot_kw=dict(polar=True))

for i in range(num_clusters):
    valores = df_grouped_norm.iloc[i].tolist()
    valores += valores[:1]

    axs[i].plot(angles, valores, color=cores[i], linewidth=2)
    axs[i].fill(angles, valores, color=cores[i], alpha=0.25)

    axs[i].set_title(
        f"{nomes_personas[i]} - {df_radar[df_radar['Cluster'] == i].shape[0]} registros",
        size=12, color=cores[i]
    )
    axs[i].set_xticks(angles[:-1])
    axs[i].set_xticklabels(labels_bonitos, fontsize=9)
    axs[i].set_yticklabels([])

# Título geral
plt.suptitle("📈 Perfil dos Clusters - Radar Chart", fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.85)

# Salvar imagem (opcional)
fig.savefig("radar_clusters.png", dpi=300)
fig.savefig("radar_clusters.pdf")

plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Aplicar KMeans com k=4
kmeans_4 = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters_4 = kmeans_4.fit_predict(dados_normalizados_norte)
df_modelo_norte["Cluster_4"] = clusters_4

# Redução de dimensionalidade para visualização
pca_4 = PCA(n_components=2)
pca_result_4 = pca_4.fit_transform(dados_normalizados_norte)

# Inércia e Silhueta para k=4
inertia_4 = kmeans_4.inertia_
silhouette_4 = silhouette_score(dados_normalizados_norte, clusters_4)

# Gráfico PCA com clusters para k=4
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(pca_result_4[:, 0], pca_result_4[:, 1], c=clusters_4, cmap='tab10', s=50)
ax.set_title(f'Visualização PCA dos Clusters (k=4)\nInércia: {inertia_4:.2f} | Silhueta: {silhouette_4:.2f}')
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.grid(True)
plt.tight_layout()
plt.show()


### 5. Avaliação (Evaluation)

A análise de agrupamentos com KMeans permitiu segmentar os municípios da região Norte com base na evolução dos seus indicadores educacionais entre 2005 e 2023.

Os clusters indicam padrões distintos de desempenho:

- **Cluster 0**: Municípios com alta evolução nos indicadores.
- **Cluster 1**: Evolução moderada.
- **Cluster 2**: Pouca ou nenhuma evolução.

A escolha de variáveis se baseia na análise longitudinal do desempenho educacional. O número de clusters foi definido com base no **método do cotovelo**, garantindo uma divisão equilibrada e coerente dos dados.

Esses resultados serão essenciais para direcionar políticas públicas mais específicas por grupo de municípios.


In [ ]:
df_analise_cluster = df_modelo_norte.groupby("cluster")[["Δ_IDEB", "Δ_Aprov"]].mean()
print(df_analise_cluster)

Implementando APRIORI AO CODIGO

In [ ]:
# Criando faixas categóricas para IDEB
df_norte['categoria_IDEB_2023'] = pd.cut(
    df_norte['IDEB_2023_4EF'],
    bins=[0, 4, 5, 6, 7, 10],
    labels=["muito_baixo", "baixo", "medio", "bom", "excelente"]
)

# Criando categoria para evolução do IDEB
df_norte['evolucao_IDEB'] = df_norte['Δ_IDEB'].apply(
    lambda x: "melhorou" if x > 0.2 else ("piorou" if x < -0.2 else "estavel")
)

# Convertendo para string (necessário para dummies)
df_norte["REDE"] = df_norte["REDE"].astype(str)


In [ ]:
df_apriori = pd.get_dummies(df_norte[["categoria_IDEB_2023", "evolucao_IDEB", "REDE", "SG_UF"]])


In [ ]:
# 1. Executa Apriori
itens_frequentes = apriori(df_apriori, min_support=0.05, use_colnames=True)

# 2. Gera regras
regras = association_rules(itens_frequentes, metric="lift", min_threshold=1.0)

# 3. Filtra regras
regras_filtradas = regras[
    (regras['lift'] >= 1.2) &
    (regras['confidence'] >= 0.6)
].sort_values(by='lift', ascending=False)


FILTRAR REGRAS UTEIS

In [ ]:
regras_filtradas = regras[
    (regras['lift'] >= 1.2) &
    (regras['confidence'] >= 0.6)
].sort_values(by='lift', ascending=False)


GRAFO DE VISUALIZAÇÃO


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()

for _, row in regras_filtradas.iterrows():
    antecedente = ', '.join(list(row['antecedents']))
    consequente = ', '.join(list(row['consequents']))
    G.add_edge(antecedente, consequente, weight=row['lift'])

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=1)
weights = [d['weight'] for (_, _, d) in G.edges(data=True)]
nx.draw(G, pos, with_labels=True, node_size=2500, font_size=9, width=weights)
plt.title("📈 Grafo de Regras de Associação")
plt.show()


## Continuidade

Não adotar apenas por ter descrição longa. Resolver esquema das colunas, sobrescrita do filtro Norte, sequência de renomeação e Cluster_4 antes de uso; substituir métricas simuladas por calculadas e alinhar escola/município e etapa escolar à base. Reaproveitar narrativa verificada.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.